In [11]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt

In [12]:
event_logs = pd.read_csv('../filtered_datasets/event_logs(1).csv')
marketing_summary = pd.read_csv('../filtered_datasets/marketing_summary(1).csv')
trend_report = pd.read_csv('../filtered_datasets/trend_report(1).csv')


In [13]:

etl_log = []

def is_usable(df, col, df_name):
    """Returns False if col is missing or 100% NaN; logs and reports."""
    if col not in df.columns or df[col].isna().all():
        etl_log.append(
            f"[ETL SKIP] {df_name}: '{col}' is missing or fully NaN. "
            f"Dependent forecast/segmentation skipped for this run."
        )
        print(f"⚠️  {df_name}.{col} unusable — skipping dependent block.")
        return False
    return True

event_time_ok = is_usable(event_logs, 'event_time', 'event_logs')
user_id_ok    = is_usable(event_logs, 'user_id', 'event_logs')
sales_ok      = is_usable(marketing_summary, 'total_sales', 'marketing_summary')
customers_ok  = is_usable(marketing_summary, 'new_customers', 'marketing_summary')

print(event_time_ok)
print(customers_ok)
print(user_id_ok )
print(sales_ok)


True
True
True
True


# 1. Customer Types

In [14]:
event_logs.head()

,user_id,event_type,event_time,product_id,amount
0,U0099,checkout,2023-06-03 04:13:00,P010,NaN
1,U0240,wishlist_add,2023-06-03 05:08:00,P020,0.0
2,U0374,profile_update,2023-06-05 06:22:00,P028,0.0
3,U0122,page_view,2023-06-06 03:45:00,P001,0.0
4,U0211,wishlist_add,2023-06-03 12:38:00,P015,0.0


In [15]:
purchase_counts = (
    event_logs[event_logs['event_type'] == 'checkout']
    .groupby('user_id')
    .size()
    .reset_index(name='purchase_count')
)

In [16]:
user_segments = (
    event_logs[['user_id']]
    .drop_duplicates()
    .merge(purchase_counts, on='user_id', how='left')
)

In [17]:
#replace missing purchases with 0
user_segments['purchase_count'] = (
    user_segments['purchase_count']
    .fillna(0)
    .astype(int)
)

In [18]:
#create segments
user_segments['segment'] = user_segments['purchase_count'].apply(
    lambda x:
        'Non Buyer' if x == 0 else
        'Single Buyer' if x == 1 else
        'Repeat Buyer'
)

In [19]:
user_segments.to_csv('user_segments.csv', index = False)

# 2-3. Trend Report Forecast and Event Log Forecast


In [20]:
if event_time_ok:
    #create time grain
    event_logs['event_time'] = pd.to_datetime(event_logs['event_time'])

    event_logs['event_hour'] = event_logs['event_time'].dt.floor('h')

    event_logs['date'] = event_logs['event_time'].dt.date
    # aggregate events by hour
    events_per_hour = (
        event_logs
        .groupby('event_hour')
        .size()
        .reset_index(name='total_events')
    )

    #aggregate users per hour
    users_per_hour = (
        event_logs
        .groupby('event_hour')['user_id']
        .nunique()
        .reset_index(name='unique_users')
    )

    #merge
    hourly_engagement = events_per_hour.merge(
        users_per_hour,
        on='event_hour',
        how='outer'
    ).sort_values('event_hour')
    
    # Write if else statement. Data could be generated so we need
    # forecasting to adjust whether or not the data is seasonal or not
    hourly = hourly_engagement.set_index('event_hour').sort_index().asfreq('h')
    hourly = hourly.fillna(0)

    forecasts = {}

    for col in ['total_events', 'unique_users']:
        p_value = adfuller(hourly[col].dropna())[1]

        if p_value >= 0.05:
            # Non-stationary → SARIMA with 24hr seasonal period
            model = SARIMAX(hourly[col], order=(1,1,1), seasonal_order=(1,0,1,24))
            print(f"{col} → SARIMA")
        else:
            # Stationary → plain ARIMA
            model = ARIMA(hourly[col], order=(1,0,1))
            print(f"{col} → ARIMA")

        result = model.fit()
        forecasts[col] = result.forecast(steps=168).round().clip(lower=0)  # forecast next 7 days

    print(forecasts)

    # assign label for tableau visualization, then merge
    actual_df = hourly_engagement.copy()
    actual_df['type'] = 'actual'

    forecast_df = pd.DataFrame({
        'event_hour': forecasts['total_events'].index,
        'total_events': forecasts['total_events'].values,
        'unique_users': forecasts['unique_users'].values,
        'type': 'forecast'
    })

    combined = pd.concat([actual_df, forecast_df], ignore_index=True)
    combined.to_csv('event_logs_forecast.csv', index=False)
 # --- Per-product hourly forecast ---
    # Aggregate by hour and product
    events_per_hour = (
    event_logs
    .groupby(['event_hour', 'product_id'])
    .size()
    .reset_index(name='total_events')
    )

    users_per_hour = (
        event_logs
        .groupby(['event_hour', 'product_id'])['user_id']
        .nunique()
        .reset_index(name='unique_users')
    )

    hourly_engagement = events_per_hour.merge(
        users_per_hour,
        on=['event_hour', 'product_id'],
        how='outer'
    ).sort_values('event_hour')
    #Loop by product. we need filters for each product on tableau, so to the members assigned in tableau, please read

    hourly_engagement['event_hour'] = pd.to_datetime(hourly_engagement['event_hour'])

    forecasts_list = []

    for product_id, group in hourly_engagement.groupby('product_id'):
        hourly = group.set_index('event_hour').sort_index().asfreq('h').fillna(0)
        
        product_forecasts = {'event_hour': pd.date_range(
            start=hourly.index[-1] + pd.Timedelta(hours=1), 
            periods=168, freq='h'
        )}
        
        for col in ['total_events', 'unique_users']:
            p_value = adfuller(hourly[col].dropna())[1]
            
            if p_value >= 0.05:
                model = SARIMAX(hourly[col], order=(1,1,1), seasonal_order=(1,0,1,24))
            else:
                model = ARIMA(hourly[col], order=(1,0,1))
            
            result = model.fit()
            product_forecasts[col] = result.forecast(steps=168).round().clip(lower=0).values
    #    
        product_forecasts['product_id'] = product_id
        forecasts_list.append(pd.DataFrame(product_forecasts))

    forecast_df = pd.concat(forecasts_list, ignore_index=True)
    forecast_df['type'] = 'forecast'
        #Export forecast

    actual_df = hourly_engagement.copy()
    actual_df['type'] = 'actual'

    combined = pd.concat([actual_df, forecast_df], ignore_index=True)

        # add date and week for Tableau filters
    combined['date'] = pd.to_datetime(combined['event_hour']).dt.date
    combined['week'] = pd.to_datetime(combined['event_hour']).dt.strftime('%Y-W%V')

    combined.to_csv('trend_report_forecast.csv', index=False)

else:
    etl_log.append("[ETL SKIP] event_logs_forecast.csv and trend_report_forecast.csv NOT generated — event_time unusable.")
    print("⚠️ Skipping both hourly forecast exports — event_time unusable.")

total_events → ARIMA


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


unique_users → ARIMA
{'total_events': 2023-06-06 20:00:00    15.0
2023-06-06 21:00:00    15.0
2023-06-06 22:00:00    15.0
2023-06-06 23:00:00    15.0
2023-06-07 00:00:00    15.0
                       ... 
2023-06-13 15:00:00    15.0
2023-06-13 16:00:00    15.0
2023-06-13 17:00:00    15.0
2023-06-13 18:00:00    15.0
2023-06-13 19:00:00    15.0
Freq: h, Name: predicted_mean, Length: 168, dtype: float64, 'unique_users': 2023-06-06 20:00:00    15.0
2023-06-06 21:00:00    15.0
2023-06-06 22:00:00    15.0
2023-06-06 23:00:00    15.0
2023-06-07 00:00:00    15.0
                       ... 
2023-06-13 15:00:00    15.0
2023-06-13 16:00:00    15.0
2023-06-13 17:00:00    15.0
2023-06-13 18:00:00    15.0
2023-06-13 19:00:00    15.0
Freq: h, Name: predicted_mean, Length: 168, dtype: float64}


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameter

RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.25786D+00    |proj g|=  9.79532D-02

At iterate    5    f=  1.19873D+00    |proj g|=  6.05194D-02

At iterate   10    f=  1.19280D+00    |proj g|=  9.16937D-03

At iterate   15    f=  1.19153D+00    |proj g|=  3.12216D-02

At iterate   20    f=  1.18770D+00    |proj g|=  1.15557D-02

At iterate   25    f=  1.18599D+00    |proj g|=  8.44591D-03

At iterate   30    f=  1.18555D+00    |proj g|=  7.76902D-03

At iterate   35    f=  1.18547D+00    |proj g|=  1.04719D-03

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function 


   evaluations in the last line search.  Termination
   may possibly be caused by a bad search direction.
 This problem is unconstrained.


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =            5     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.25786D+00    |proj g|=  9.79532D-02

At iterate    5    f=  1.19873D+00    |proj g|=  6.05194D-02

At iterate   10    f=  1.19280D+00    |proj g|=  9.16937D-03

At iterate   15    f=  1.19153D+00    |proj g|=  3.12216D-02

At iterate   20    f=  1.18770D+00    |proj g|=  1.15557D-02

At iterate   25    f=  1.18599D+00    |proj g|=  8.44591D-03

At iterate   30    f=  1.18555D+00    |proj g|=  7.76902D-03

At iterate   35    f=  1.18547D+00    |proj g|=  1.04719D-03

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function 


   evaluations in the last line search.  Termination
   may possibly be caused by a bad search direction.
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/miniconda3/envs/ml/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible sta

# 4. Marketing Summary Forecast

In [22]:
if sales_ok and customers_ok:
    #create key columns for visualization metrics
    marketing_summary['date'] = pd.to_datetime(marketing_summary['date'])
    marketing_summary = marketing_summary.sort_values('date').set_index('date')
    marketing_summary = marketing_summary.asfreq('D').fillna(0)

    #forecast them both for 2 forecasts (see canva link for reference if you're making the Tableau charts)
    
    marketing_forecasts = {}

    for col in ['new_customers', 'total_sales']:
        p_value = adfuller(marketing_summary[col].dropna())[1]
        
        if p_value >= 0.05:
            model = SARIMAX(marketing_summary[col], order=(1,1,1), seasonal_order=(1,0,1,7))
            print(f"{col} → SARIMA")
        else:
            model = ARIMA(marketing_summary[col], order=(1,0,1))
            print(f"{col} → ARIMA")
        
        result = model.fit()
        marketing_forecasts[col] = result.forecast(steps=7).round().clip(lower=0)

    print(marketing_forecasts)
    #label and export

    actual_df = marketing_summary[['new_customers', 'total_sales']].copy()
    actual_df['type'] = 'actual'
    actual_df = actual_df.reset_index()  # brings date back as column

    forecast_df = pd.DataFrame({
        'date': marketing_forecasts['new_customers'].index,
        'new_customers': marketing_forecasts['new_customers'].values,
        'total_sales': marketing_forecasts['total_sales'].values,
        'type': 'forecast'
    })

    combined_marketing = pd.concat([actual_df, forecast_df], ignore_index=True)

    # add week column for Tableau filter
    combined_marketing['week'] = pd.to_datetime(combined_marketing['date']).dt.strftime('%Y-W%V')

    combined_marketing.to_csv('marketing_summary_forecast.csv', index=False)
    print(combined_marketing)

else:
    etl_log.append("[ETL SKIP] marketing_summary_forecast.csv NOT generated — total_sales or new_customers unusable.")
    print("⚠️ Skipping marketing forecast export — total_sales or new_customers unusable.")

new_customers → ARIMA
total_sales → ARIMA
{'new_customers': 2023-09-09    8.0
2023-09-10    8.0
2023-09-11    8.0
2023-09-12    8.0
2023-09-13    8.0
2023-09-14    8.0
2023-09-15    8.0
Freq: D, Name: predicted_mean, dtype: float64, 'total_sales': 2023-09-09    60347.0
2023-09-10    58721.0
2023-09-11    58085.0
2023-09-12    57836.0
2023-09-13    57739.0
2023-09-14    57700.0
2023-09-15    57685.0
Freq: D, Name: predicted_mean, dtype: float64}
          date  new_customers  total_sales      type      week
0   2023-06-01            9.0     81287.31    actual  2023-W22
1   2023-06-02            5.0     74771.99    actual  2023-W22
2   2023-06-03           11.0     84809.74    actual  2023-W22
3   2023-06-04            3.0     61212.30    actual  2023-W22
4   2023-06-05           10.0     80911.49    actual  2023-W23
..         ...            ...          ...       ...       ...
102 2023-09-11            8.0     58085.00  forecast  2023-W37
103 2023-09-12            8.0     57836.00  for

In [23]:
print("\n--- ETL Run Summary ---")
if etl_log:
    print("\n".join(etl_log))
else:
    print("All forecast blocks ran successfully — no unusable columns detected.")


--- ETL Run Summary ---
All forecast blocks ran successfully — no unusable columns detected.
